In [55]:
from utils.data_prep import split_data, calculate_conflict_ratio
from utils.dates import *
from utils.cross_validation import (
    grouped_timeseries_cv_ids,
    verify_cv_splits,
    timeseries_cross_val_predict,
)
from utils.data_prep import get_clean_combined_data
import numpy as np
import pandas as pd
import xgboost as xgb
import logging

from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
)
from sklearn.model_selection import RandomizedSearchCV

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("Train NLP")
logger.setLevel(logging.INFO)


In [56]:
DOWNLOAD = False
REMOVE_ABYEI = False

In [17]:
# def get_pcs(df):
#     emb_cols = [c for c in df.columns if c.startswith("emb_")]
#     X = df[emb_cols].values
#
#     pca = PCA()
#     pca.fit(X)
#
#     cumulative_variance = pca.explained_variance_ratio_.cumsum()
#     n_components_90 = (cumulative_variance < 0.90).sum() + 1
#     pca_optimal = PCA(n_components=n_components_90)
#     X_pcs = pca_optimal.fit_transform(X)
#
#     logger.info(f"Components needed for 90% variance: {n_components_90}")
#
#     pc_column_names = [f"PC{i+1}" for i in range(n_components_90)]
#     df_pcs = pd.DataFrame(X_pcs, columns=pc_column_names, index=df.index)
#
#     metadata_cols = [c for c in df.columns if not c.startswith("emb_")]
#     df_metadata = df[metadata_cols]
#
#     df_final_pcs = pd.concat([df_metadata, df_pcs], axis=1)
#     df_final_pcs = df_final_pcs.rename(columns={"admin1": "region"})
#
#     return df_final_pcs

In [50]:
def apply_pca_train_only(train_df, onset_df, active_df, predictor_cols, variance_threshold=0.90):

    emb_cols = [c for c in predictor_cols if c.startswith("emb_")]
    non_emb_cols = [c for c in predictor_cols if not c.startswith("emb_")]

    pca = PCA(n_components=variance_threshold, random_state=7)

    X_train_emb_pca = pca.fit_transform(train_df[emb_cols])
    X_onset_emb_pca = pca.transform(onset_df[emb_cols])
    X_active_emb_pca = pca.transform(active_df[emb_cols])

    logger.info(
        f"PCA fit on train embeddings only: {len(emb_cols)} raw dims -> "
        f"{pca.n_components_} components for {variance_threshold:.0%} variance."
    )

    pc_names = [f"PC{i+1}" for i in range(pca.n_components_)]

    def rebuild(df, emb_pca_array):
        pcs = pd.DataFrame(emb_pca_array, columns=pc_names, index=df.index)
        return pd.concat([df[non_emb_cols].reset_index(drop=True),
                           pcs.reset_index(drop=True)], axis=1)

    X_train = rebuild(train_df, X_train_emb_pca)
    X_onset = rebuild(onset_df, X_onset_emb_pca)
    X_active = rebuild(active_df, X_active_emb_pca)

    final_predictor_cols = non_emb_cols + pc_names
    return X_train, X_onset, X_active, pca, final_predictor_cols


In [53]:
def train_evaluate_model(processed_df, predictor_cols, params, best_params=False, use_pca=True):

    # Still includes emb
    train_df, y_train, _ = split_data(
        processed_df, predictor_cols, train_start_date, train_end_date,
    )
    onset_df, y_onset, _ = split_data(
        processed_df, predictor_cols, onset_start_date, onset_end_date,
    )
    active_df, y_active, _ = split_data(
        processed_df, predictor_cols, active_start_date, active_end_date,
    )

    if use_pca:
        X_train, X_onset, X_active, pca, final_predictor_cols = apply_pca_train_only(
            train_df, onset_df, active_df, predictor_cols
        )
    else:
        X_train = train_df[predictor_cols].copy()
        X_onset = onset_df[predictor_cols].copy()
        X_active = active_df[predictor_cols].copy()
        final_predictor_cols = predictor_cols

    X_train.columns = X_train.columns.astype(object)
    X_onset.columns = X_onset.columns.astype(object)
    X_active.columns = X_active.columns.astype(object)

    ratios = calculate_conflict_ratio(train_df)
    scale_weight = ratios["non-escalation"] / ratios["escalation"]

    n_splits = params.get("n_splits", 4)
    grouped_timeseries_cv = list(
        grouped_timeseries_cv_ids(train_df["year_month"], n_splits=n_splits)
    )
    verify_cv_splits(train_df, grouped_timeseries_cv)

    param_grid = {
        k: v
        for k, v in params.items()
        if k not in ["k", "event_col", "n_splits", "remove_abyei"]
    }

    if best_params:
        best_model = xgb.XGBClassifier(
            scale_pos_weight=scale_weight,
            eval_metric="aucpr",
            random_state=7,
            **param_grid,
        )
        best_model.fit(X_train, y_train)
        fitted_best_params = params
    else:
        xgb_model = xgb.XGBClassifier(
            scale_pos_weight=scale_weight,
            eval_metric="aucpr",
            random_state=7,
        )

        random_search = RandomizedSearchCV(
            estimator=xgb_model,
            param_distributions=param_grid,
            n_iter=150,
            cv=grouped_timeseries_cv,
            scoring="average_precision",
            n_jobs=-1,
            random_state=23,
        )

        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_
        fitted_best_params = random_search.best_params_

    oof_y_true, oof_y_proba = timeseries_cross_val_predict(
        best_model, X_train, y_train, grouped_timeseries_cv
    )

    precisions, recalls, thresholds = precision_recall_curve(oof_y_true, oof_y_proba)
    f1_scores = (2 * precisions * recalls / (precisions + recalls + 1e-10))[:-1]
    optimal_threshold = thresholds[np.argmax(f1_scores)]

    # Evaluate on onset test set
    y_pred_proba_onset = best_model.predict_proba(X_onset)[:, 1]
    y_pred_custom_onset = (y_pred_proba_onset >= optimal_threshold).astype(int)

    # Evaluate on active test set
    y_pred_proba_active = best_model.predict_proba(X_active)[:, 1]
    y_pred_custom_active = (y_pred_proba_active >= optimal_threshold).astype(int)

    onset_report = classification_report(
        y_onset, y_pred_custom_onset, output_dict=True, zero_division=0
    )
    active_report = classification_report(
        y_active, y_pred_custom_active, output_dict=True, zero_division=0
    )

    class_key = "1" if "1" in onset_report else 1

    results = {
        "optimal_threshold": f"{optimal_threshold:.4f}",
        "n_predictors": len(final_predictor_cols),
        # Onset Metrics
        "onset_aupr": f"{average_precision_score(y_onset, y_pred_proba_onset):.4f}",
        "onset_precision_class1": f"{onset_report[class_key]['precision']:.4f}",
        "onset_recall_class1": f"{onset_report[class_key]['recall']:.4f}",
        "onset_f1_class1": f"{onset_report[class_key]['f1-score']:.4f}",
        # Active Metrics
        "active_aupr": f"{average_precision_score(y_active, y_pred_proba_active):.4f}",
        "active_precision_class1": f"{active_report[class_key]['precision']:.4f}",
        "active_recall_class1": f"{active_report[class_key]['recall']:.4f}",
        "active_f1_class1": f"{active_report[class_key]['f1-score']:.4f}",
    }

    fitted_best_params["k"] = params["k"]
    fitted_best_params["n_splits"] = params["n_splits"]
    fitted_best_params["event_col"] = params["event_col"]

    return results, fitted_best_params


In [54]:
k = 0.75
n = 4
event_col = "event_type"
data_sources = ["rain", "food", "notes"]

test_params = {
    "max_depth": 3,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0,
    "reg_lambda": 1,
    "colsample_bylevel": 0.8,
    "k": k,
    "event_col": event_col,
    "n_splits": n,
    "remove_abyei": REMOVE_ABYEI,
}

model_data, predictor_cols = get_clean_combined_data(
    test_params,
    data_sources=data_sources,
    download=DOWNLOAD,
    remove_abyei=REMOVE_ABYEI,
)


# With PCA
results_pca, params_pca = train_evaluate_model(
    model_data, predictor_cols, test_params, best_params=True, use_pca=True
)
logger.info(f"Results WITH PCA: {results_pca}")

# Without PCA
results_raw, params_raw = train_evaluate_model(
    model_data, predictor_cols, test_params, best_params=True, use_pca=False
)
logger.info(f"Results WITHOUT PCA: {results_raw}")

INFO:ACLED processing:Data grouped by event_type
INFO:ACLED processing:Escalation target set at 0.75 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv
INFO:Data preparation:Food prices data processed.
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Data preparation:Rainfall data processed.
INFO:Data preparation:Notes data processed.
INFO:Train NLP:PCA fit on train embeddings only: 768 raw dims -> 100 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------


INFO:Train NLP:Results WITH PCA: {'optimal_threshold': '0.3740', 'n_predictors': 114, 'onset_aupr': '0.6441', 'onset_precision_class1': '0.6140', 'onset_recall_class1': '0.4667', 'onset_f1_class1': '0.5303', 'active_aupr': '0.5011', 'active_precision_class1': '0.5833', 'active_recall_class1': '0.3088', 'active_f1_class1': '0.4038'}
INFO:Cross validation:Cross-validation testing splits:


--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------


INFO:Train NLP:Results WITHOUT PCA: {'optimal_threshold': '0.3403', 'n_predictors': 782, 'onset_aupr': '0.7245', 'onset_precision_class1': '0.6056', 'onset_recall_class1': '0.5733', 'onset_f1_class1': '0.5890', 'active_aupr': '0.5093', 'active_precision_class1': '0.5094', 'active_recall_class1': '0.3971', 'active_f1_class1': '0.4463'}
